In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
import gymnasium as gym

from gymnasium import spaces

import numpy as np

import matplotlib.pyplot as plt

from stable_baselines3 import PPO

from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

import torch
import torch.nn as nn

In [ ]:
class CustomCNN(BaseFeaturesExtractor):

    def __init__(self, observation_space, features_dim=256):

        super(CustomCNN, self).__init__(
            observation_space,
            features_dim
        )

        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(

            nn.Conv2d(
                n_input_channels,
                32,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            nn.Flatten()
        )

        with torch.no_grad():

            sample_input = torch.as_tensor(
                observation_space.sample()[None]
            ).float()

            n_flatten = self.cnn(sample_input).shape[1]

        self.linear = nn.Sequential(

            nn.Linear(n_flatten, features_dim),

            nn.ReLU()
        )

    def forward(self, observations):

        return self.linear(
            self.cnn(observations)
        )

In [ ]:
class SaudiWildfireEnv(gym.Env):

    def __init__(

        self,

        use_wind=True,

        use_terrain=True,

        use_suppression=True,

        dense_fuel=False
    ):

        super(SaudiWildfireEnv, self).__init__()

        # =====================================
        # CONFIGURATION FLAGS
        # =====================================

        self.use_wind = use_wind

        self.use_terrain = use_terrain

        self.use_suppression = use_suppression

        self.dense_fuel = dense_fuel

        # =====================================
        # LOAD TENSOR
        # =====================================

        tensor_path = (
            "/content/drive/MyDrive/"
            "PyroRL_Saudi_Project/datasets/"
            "saudi_eastern_province/grids/32x32/"
            "state_tensor.npy"
        )

        self.initial_state = np.load(tensor_path)

        self.state = self.initial_state.copy()

        # =====================================
        # DENSE FUEL OPTION
        # =====================================

        if self.dense_fuel:

            self.state[1] = np.clip(
                self.state[1] * 2.0,
                0,
                1
            )

        # =====================================
        # SPACES
        # =====================================

        self.observation_space = spaces.Box(
            low=0,
            high=1,
            shape=(7,32,32),
            dtype=np.float32
        )

        self.action_space = spaces.Discrete(5)

        self.agent_pos = [16,16]

        self.current_step = 0

        self.max_steps = 200

In [ ]:
class SaudiWildfireEnv(gym.Env):
    def __init__(self, use_wind=True, use_terrain=True, use_suppression=True, dense_fuel=False):
        super(SaudiWildfireEnv, self).__init__()
        # =====================================
        # CONFIGURATION FLAGS
        # =====================================
        self.use_wind = use_wind
        self.use_terrain = use_terrain
        self.use_suppression = use_suppression
        self.dense_fuel = dense_fuel

        # =====================================
        # LOAD STATE TENSOR
        # =====================================
        tensor_path = (
            "/content/drive/MyDrive/"
            "PyroRL_Saudi_Project/datasets/"
            "saudi_eastern_province/grids/32x32/"
            "state_tensor.npy"
        )
        self.initial_state = np.load(tensor_path)
        self.state = self.initial_state.copy()

        if self.dense_fuel:
            self.state[1] = np.clip(self.state[1] * 2.0, 0, 1)

        print("Loaded state shape:", self.state.shape)

        # =====================================
        # OBSERVATION SPACE
        # =====================================
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(7, 32, 32),
            dtype=np.float32
        )

        # =====================================
        # ACTION SPACE
        # =====================================
        self.action_space = spaces.Discrete(5)

        # =====================================
        # AGENT POSITION
        # =====================================
        self.agent_pos = [16, 16]

        # =====================================
        # STEP COUNTER
        # =====================================
        self.current_step = 0
        self.max_steps = 200

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.initial_state.copy()
        if self.dense_fuel:
            self.state[1] = np.clip(self.state[1] * 2.0, 0, 1)
        self.agent_pos = [16, 16]
        self.current_step = 0
        return self.state.astype(np.float32), {}

    def step(self, action):
        x, y = self.agent_pos
        # MOVEMENT
        if action == 0: x -= 1
        elif action == 1: x += 1
        elif action == 2: y -= 1
        elif action == 3: y += 1

        x = np.clip(x, 0, 31)
        y = np.clip(y, 0, 31)
        self.agent_pos = [x, y]

        # FIRE SUPPRESSION
        fire_layer = self.state[0]
        if self.use_suppression:
            suppression_power = 0.05
            fire_layer[x, y] = max(0, fire_layer[x, y] - suppression_power)

        # SPATIAL FIRE SPREAD WITH DIRECTIONAL WIND AND TERRAIN
        fuel_layer = self.state[1]
        wind_x = self.state[2]
        wind_y = self.state[3]
        terrain_layer = self.state[4]
        new_fire_layer = fire_layer.copy()

        for i in range(1, 31):
            for j in range(1, 31):
                current_fire = fire_layer[i, j]
                if current_fire < 0.05: continue

                # STEP 3: Directional Spread Weights
                neighbors = [
                    ((i-1, j), 1.0),  # up
                    ((i+1, j), 1.0),  # down
                    ((i, j-1), 1.0),  # left
                    ((i, j+1), 1.0)   # right
                ]

                for (ni, nj), direction_weight in neighbors:
                    fuel = fuel_layer[ni, nj]
                    local_wind_x = wind_x[i, j]
                    local_wind_y = wind_y[i, j]

                    # Directional wind influence
                    wind_bonus = 1.0
                    if self.use_wind:
                        if local_wind_x > 0 and ni > i: wind_bonus += abs(local_wind_x)
                        if local_wind_x < 0 and ni < i: wind_bonus += abs(local_wind_x)
                        if local_wind_y > 0 and nj > j: wind_bonus += abs(local_wind_y)
                        if local_wind_y < 0 and nj < j: wind_bonus += abs(local_wind_y)

                    # STEP 5: Terrain Influence
                    terrain_factor = 1.0
                    if self.use_terrain:
                        terrain_factor += terrain_layer[ni, nj]

                    spread_amount = (
                        0.02
                        * current_fire
                        * fuel
                        * wind_bonus
                        * terrain_factor
                    )
                    new_fire_layer[ni, nj] += spread_amount

        fire_layer = np.clip(new_fire_layer, 0, 1)
        self.state[0] = fire_layer

        # FUEL CONSUMPTION
        burn_rate = 0.01
        fuel_layer = np.clip(fuel_layer - burn_rate * fire_layer, 0, 1)

        # STEP 6: Add Burn Scars
        burned_mask = fire_layer > 0.7
        fuel_layer[burned_mask] *= 0.5

        self.state[1] = fuel_layer

        # STEP 7: REWARD FUNCTION IMPROVEMENT
        mean_fire = np.mean(fire_layer)
        reward = -float(mean_fire)

        # Suppression bonus (active action)
        if fire_layer[x, y] > 0:
            reward += 1.0

        # Bonus for overall containment
        if mean_fire < 0.2:
            reward += 2.0

        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False
        return self.state.astype(np.float32), reward, terminated, truncated, {}

In [ ]:
def run_experiment(

    experiment_name,

    use_wind=True,

    use_terrain=True,

    use_suppression=True,

    dense_fuel=False,

    timesteps=10000
):

    env = SaudiWildfireEnv(

        use_wind=use_wind,

        use_terrain=use_terrain,

        use_suppression=use_suppression,

        dense_fuel=dense_fuel
    )

    policy_kwargs = dict(

        features_extractor_class=CustomCNN,

        features_extractor_kwargs=dict(
            features_dim=256
        )
    )

    model = PPO(

        "CnnPolicy",

        env,

        policy_kwargs=policy_kwargs,

        verbose=0
    )

    model.learn(total_timesteps=timesteps)

    fire_values = []

    obs, info = env.reset()

    for _ in range(200):

        action, _ = model.predict(obs)

        obs, reward, done, truncated, info = env.step(action)

        fire_values.append(
            np.mean(env.state[0])
        )

        if done:
            break

    return fire_values

In [ ]:
baseline_fire = run_experiment(

    "Baseline",

    use_wind=True,

    use_terrain=True,

    use_suppression=True,

    dense_fuel=False
)

print("Baseline experiment complete.")

Run No-Wind Experiment 🌬️

In [ ]:
no_wind_fire = run_experiment(

    "No Wind",

    use_wind=False,

    use_terrain=True,

    use_suppression=True,

    dense_fuel=False
)

print("No-wind experiment complete.")

Run No-Terrain Experiment ⛰️

In [ ]:
no_terrain_fire = run_experiment(

    "No Terrain",

    use_wind=True,

    use_terrain=False,

    use_suppression=True,

    dense_fuel=False
)

print("No-terrain experiment complete.")

Run No-Suppression Experiment 🚒

In [ ]:
no_suppression_fire = run_experiment(

    "No Suppression",

    use_wind=True,

    use_terrain=True,

    use_suppression=False,

    dense_fuel=False
)

print("No-suppression experiment complete.")

Run Dense Fuel Experiment 🌲

In [ ]:
dense_fuel_fire = run_experiment(

    "Dense Fuel",

    use_wind=True,

    use_terrain=True,

    use_suppression=True,

    dense_fuel=True
)

print("Dense-fuel experiment complete.")

Plot ALL Experiments 🔥

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    baseline_fire,
    label="Baseline"
)

plt.plot(
    no_wind_fire,
    label="No Wind"
)

plt.plot(
    no_terrain_fire,
    label="No Terrain"
)

plt.plot(
    no_suppression_fire,
    label="No Suppression"
)

plt.plot(
    dense_fuel_fire,
    label="Dense Fuel"
)

plt.xlabel("Step")

plt.ylabel("Mean Fire Intensity")

plt.title("Saudi Wildfire Ablation Experiments")

plt.legend()

plt.grid()

plt.show()

Compute Final Metrics

In [ ]:
results = {

    "Baseline":
        baseline_fire[-1],

    "No Wind":
        no_wind_fire[-1],

    "No Terrain":
        no_terrain_fire[-1],

    "No Suppression":
        no_suppression_fire[-1],

    "Dense Fuel":
        dense_fuel_fire[-1]
}

results

Create Results Table

In [ ]:
for experiment, value in results.items():

    print(
        f"{experiment}: "
        f"{value:.4f}"
    )

Save Results

In [ ]:
np.save(

    "/content/drive/MyDrive/"
    "PyroRL_Saudi_Project/"
    "phase6_results.npy",

    results
)

print("Results saved.")